<a href="https://colab.research.google.com/github/hemel5612-lgtm/Aleurodicus-rugioperculatus/blob/main/Aleurodicus_rugioperculatus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# STEP 0 — Project setup, workbook audit, secure GitHub sync
# Tables -> XLSX only
# Figures -> PNG only
# Token is NOT stored in git remote URL
# ============================================================

# -----------------------------
# 0.1 Mount Google Drive
# -----------------------------
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# -----------------------------
# 0.2 Imports
# -----------------------------
import os
import re
import glob
import json
import shutil
import warnings
import subprocess
import sys
import base64
from pathlib import Path
from getpass import getpass

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

try:
    import nbformat as nbf
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "nbformat"], check=True)
    import nbformat as nbf

# -----------------------------
# 0.3 User paths
# -----------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/Data Analysis March 2026")
OUTPUT_DIR = PROJECT_ROOT / "Mishu"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_XLSX = PROJECT_ROOT / "Aleurodicus_rugioperculatus.xlsx"
TARGET_AUDIT_XLSX = OUTPUT_DIR / "Step0_Workbook_Audit.xlsx"
TARGET_META_JSON = OUTPUT_DIR / "Step0_Project_Metadata.json"
CLEANED_WORKBOOK_COPY = OUTPUT_DIR / "Step0_Cleaned_Workbook_Copy.xlsx"

print(f"Project root : {PROJECT_ROOT}")
print(f"Output folder: {OUTPUT_DIR}")

# -----------------------------
# 0.4 Config flags
# -----------------------------
AUTO_PUSH_STEP0 = True
SAVE_CLEANED_WORKBOOK_COPY = True

# -----------------------------
# 0.5 Helpers
# -----------------------------
def run_cmd(cmd, cwd=None, check=True, quiet=False, env=None):
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True, env=env)
    if check and result.returncode != 0:
        print("\nSTDOUT:\n", result.stdout)
        print("\nSTDERR:\n", result.stderr)
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    if not quiet:
        if result.stdout.strip():
            print(result.stdout.strip())
        if result.stderr.strip():
            print(result.stderr.strip())
    return result

def get_secret_value(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return None

def prompt_if_missing(current_value, prompt_text, secret=False, allow_blank=True):
    if current_value:
        return current_value
    if secret:
        value = getpass(prompt_text)
    else:
        value = input(prompt_text)
    value = value.strip()
    if not value and allow_blank:
        return None
    return value

def find_candidate_files(exact_name=None, fuzzy_patterns=None, search_roots=None, max_hits=300):
    if search_roots is None:
        search_roots = [
            "/content",
            "/content/drive/MyDrive",
            "/mnt/data",
        ]

    candidates = []

    if exact_name:
        for root in search_roots:
            candidates.extend(glob.glob(os.path.join(root, "**", exact_name), recursive=True))

    if fuzzy_patterns:
        for fp in fuzzy_patterns:
            for root in search_roots:
                candidates.extend(glob.glob(os.path.join(root, "**", fp), recursive=True))

    seen = set()
    clean = []
    for p in candidates:
        if p not in seen and os.path.isfile(p):
            seen.add(p)
            clean.append(p)

    return clean[:max_hits]

def extract_github_ipynb_url(text):
    if not isinstance(text, str):
        return None
    pattern = r"https?://github\.com/[^\s]+/blob/[^\s]+\.ipynb"
    match = re.search(pattern, text.strip())
    return match.group(0) if match else None

def parse_github_blob_url(url):
    pattern = r"https?://github\.com/([^/]+)/([^/]+)/blob/([^/]+)/(.+)"
    match = re.match(pattern, url)
    if not match:
        raise ValueError(f"Could not parse GitHub blob URL:\n{url}")
    owner, repo, branch, relpath = match.groups()
    return {
        "owner": owner,
        "repo": repo,
        "branch": branch,
        "relpath": relpath,
        "repo_url": f"https://github.com/{owner}/{repo}.git",
        "raw_url": f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/{relpath}",
    }

def github_auth_header(token):
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    return f"AUTHORIZATION: basic {auth}"

def git_authenticated_cmd(git_args, token, cwd=None, quiet=False, check=False):
    if not token:
        return run_cmd(["git", *git_args], cwd=cwd, quiet=quiet, check=check)
    header = github_auth_header(token)
    cmd = ["git", "-c", f"http.https://github.com/.extraheader={header}", *git_args]
    return run_cmd(cmd, cwd=cwd, quiet=quiet, check=check)

def sanitize_remote_url(url):
    return url.replace("https://", "https://***@") if "@" in url else url

# -----------------------------
# 0.6 Workbook discovery
# -----------------------------
xlsx_candidates = find_candidate_files(
    exact_name="Aleurodicus_rugioperculatus.xlsx",
    fuzzy_patterns=[
        "*Aleurodicus*.xlsx",
        "*rugioperculatus*.xlsx",
        "*.xlsx",
    ]
)

if not xlsx_candidates:
    raise FileNotFoundError(
        "No Excel workbook was found.\n\n"
        "Upload the workbook to Colab or place it in Drive, then rerun Step 0."
    )

print("\nCandidate Excel files found:")
for p in xlsx_candidates[:20]:
    print(" -", p)

chosen_xlsx = None
for p in xlsx_candidates:
    if os.path.basename(p) == "Aleurodicus_rugioperculatus.xlsx":
        chosen_xlsx = p
        break

if chosen_xlsx is None:
    chosen_xlsx = xlsx_candidates[0]

print(f"\nUsing workbook:\n{chosen_xlsx}")

if Path(chosen_xlsx).resolve() != TARGET_XLSX.resolve():
    shutil.copy2(chosen_xlsx, TARGET_XLSX)
    print(f"Workbook copied to:\n{TARGET_XLSX}")
else:
    print("Workbook already present in the project folder.")

# -----------------------------
# 0.7 GitHub notebook URL
# Priority:
# 1) env/secret GITHUB_NOTEBOOK_URL
# 2) uploaded Github-Link.txt / similar
# 3) prompt
# Excludes analysis_outputs to avoid self-reference
# -----------------------------
github_url = get_secret_value("GITHUB_NOTEBOOK_URL")
github_source_file = "env_or_secret" if github_url else None

if not github_url:
    txt_candidates = find_candidate_files(
        exact_name="Github-Link.txt",
        fuzzy_patterns=["*Github*.txt", "*GitHub*.txt", "*github*.txt"],
        search_roots=["/mnt/data", "/content/drive/MyDrive", "/content"]
    )

    filtered_txt_candidates = []
    for p in txt_candidates:
        norm = p.replace("\\", "/")
        if "/analysis_outputs/" in norm:
            continue
        filtered_txt_candidates.append(p)

    print("\nCandidate text files checked for notebook URL:")
    for p in filtered_txt_candidates[:20]:
        print(" -", p)

    preferred_txt = [
        p for p in filtered_txt_candidates
        if os.path.basename(p).lower() in ["github-link.txt", "github link.txt"]
    ]
    other_txt = [p for p in filtered_txt_candidates if p not in preferred_txt]

    for candidate_list in [preferred_txt, other_txt]:
        for candidate in candidate_list:
            try:
                with open(candidate, "r", encoding="utf-8", errors="ignore") as f:
                    txt = f.read()
                found_url = extract_github_ipynb_url(txt)
                if found_url:
                    github_url = found_url
                    github_source_file = candidate
                    break
            except Exception:
                pass
        if github_url:
            break

if not github_url:
    github_url = input("\nPaste the GitHub notebook blob URL:\n").strip()
    github_source_file = "interactive_input"

print("\nGitHub notebook reference:")
print(github_url)
print("GitHub URL source:")
print(github_source_file)

gh = parse_github_blob_url(github_url)

print("\nParsed GitHub info:")
for k, v in gh.items():
    print(f" - {k}: {v}")

# -----------------------------
# 0.8 Git credentials
# -----------------------------
GITHUB_TOKEN = get_secret_value("GITHUB_TOKEN")
GIT_USER_NAME = get_secret_value("GIT_USER_NAME")
GIT_USER_EMAIL = get_secret_value("GIT_USER_EMAIL")

print("\nGit configuration")
print("Leave token blank if you only want local commits.")

GIT_USER_NAME = prompt_if_missing(GIT_USER_NAME, "Git user name: ", secret=False, allow_blank=False)
GIT_USER_EMAIL = prompt_if_missing(GIT_USER_EMAIL, "Git user email: ", secret=False, allow_blank=False)
GITHUB_TOKEN = prompt_if_missing(GITHUB_TOKEN, "GitHub token (hidden, optional): ", secret=True, allow_blank=True)

REPO_LOCAL_DIR = Path("/content") / gh["repo"]
REPO_OUTPUTS_DIR = REPO_LOCAL_DIR / "analysis_outputs"
NOTEBOOK_REPO_PATH = REPO_LOCAL_DIR / gh["relpath"]

# -----------------------------
# 0.9 Clone or update repo
# Keep remote URL plain, no token embedded
# -----------------------------
if REPO_LOCAL_DIR.exists() and (REPO_LOCAL_DIR / ".git").exists():
    print(f"\nRepo already exists locally:\n{REPO_LOCAL_DIR}")
    run_cmd(["git", "fetch", "origin"], cwd=REPO_LOCAL_DIR, quiet=True, check=False)
    run_cmd(["git", "checkout", gh["branch"]], cwd=REPO_LOCAL_DIR, quiet=True, check=False)
    run_cmd(["git", "pull", "origin", gh["branch"]], cwd=REPO_LOCAL_DIR, quiet=True, check=False)
else:
    if REPO_LOCAL_DIR.exists():
        shutil.rmtree(REPO_LOCAL_DIR)
    print(f"\nCloning repo into:\n{REPO_LOCAL_DIR}")
    run_cmd(["git", "clone", "--branch", gh["branch"], gh["repo_url"], str(REPO_LOCAL_DIR)], quiet=False)

# Always set desired git author for this repo
run_cmd(["git", "config", "user.name", GIT_USER_NAME], cwd=REPO_LOCAL_DIR, quiet=True, check=False)
run_cmd(["git", "config", "user.email", GIT_USER_EMAIL], cwd=REPO_LOCAL_DIR, quiet=True, check=False)

# Always keep origin clean/plain
run_cmd(["git", "remote", "set-url", "origin", gh["repo_url"]], cwd=REPO_LOCAL_DIR, quiet=True, check=False)

print("\nGitHub sync status:")
print(" - Local repo path :", REPO_LOCAL_DIR)
print(" - Notebook path   :", NOTEBOOK_REPO_PATH)
print(" - Push enabled    :", bool(GITHUB_TOKEN))
print(" - Git user name   :", GIT_USER_NAME)
print(" - Git user email  :", GIT_USER_EMAIL)

print("\nGit remote check:")
run_cmd(["git", "remote", "-v"], cwd=REPO_LOCAL_DIR, quiet=False, check=False)

# -----------------------------
# 0.10 Ensure notebook exists
# -----------------------------
NOTEBOOK_REPO_PATH.parent.mkdir(parents=True, exist_ok=True)

if not NOTEBOOK_REPO_PATH.exists():
    nb = nbf.v4.new_notebook()
    nb.cells = [
        nbf.v4.new_markdown_cell(
            "# Aleurodicus rugioperculatus analysis\n\n"
            "This notebook is synchronized from Colab.\n"
            "All tables are exported as XLSX and all figures as PNG in `analysis_outputs/step_xx/`."
        )
    ]
    with open(NOTEBOOK_REPO_PATH, "w", encoding="utf-8") as f:
        nbf.write(nb, f)
    print("\nNotebook scaffold created.")
else:
    try:
        with open(NOTEBOOK_REPO_PATH, "r", encoding="utf-8") as f:
            nb_existing = nbf.read(f, as_version=4)
        print(f"\nNotebook already exists in repo with {len(nb_existing.cells)} cells.")
    except Exception:
        print("\nNotebook exists but could not be parsed with nbformat.")

# -----------------------------
# 0.11 Repo output helpers
# -----------------------------
REPO_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
STEP0_REPO_DIR = REPO_OUTPUTS_DIR / "step_00"
STEP0_REPO_DIR.mkdir(parents=True, exist_ok=True)

def repo_step_dir(step_number):
    step_dir = REPO_OUTPUTS_DIR / f"step_{int(step_number):02d}"
    step_dir.mkdir(parents=True, exist_ok=True)
    return step_dir

def save_df_to_repo(df, step_number, filename, index=False):
    if not filename.lower().endswith(".xlsx"):
        raise ValueError("Tables must be saved as .xlsx files only.")
    out_dir = repo_step_dir(step_number)
    out_path = out_dir / filename
    df.to_excel(out_path, index=index)
    print(f"Saved table to: {out_path}")
    return out_path

def save_text_to_repo(text, step_number, filename):
    out_dir = repo_step_dir(step_number)
    out_path = out_dir / filename
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(str(text))
    print(f"Saved text to: {out_path}")
    return out_path

def save_figure_to_repo(fig, step_number, stem, dpi=300):
    out_dir = repo_step_dir(step_number)
    out_path = out_dir / f"{stem}.png"
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    print(f"Saved figure to: {out_path}")
    return out_path

def _repo_relpath(path_obj):
    path_obj = Path(path_obj)
    if path_obj.is_absolute():
        return str(path_obj.relative_to(REPO_LOCAL_DIR))
    return str(path_obj)

def git_status():
    result = run_cmd(["git", "status", "--short"], cwd=REPO_LOCAL_DIR, quiet=True, check=False)
    print(result.stdout if result.stdout.strip() else "Working tree clean.")
    return result.stdout

def git_commit_push(message, add_paths=None):
    if add_paths is None:
        add_paths = ["."]
    if isinstance(add_paths, (str, Path)):
        add_paths = [add_paths]

    repo_paths = [_repo_relpath(p) for p in add_paths]
    run_cmd(["git", "add", *repo_paths], cwd=REPO_LOCAL_DIR, quiet=True, check=False)

    status = run_cmd(["git", "status", "--porcelain"], cwd=REPO_LOCAL_DIR, quiet=True, check=False)
    if not status.stdout.strip():
        print("No changes to commit.")
        return {
            "committed": False,
            "pushed": False,
            "reason": "no_changes"
        }

    commit_res = run_cmd(["git", "commit", "-m", message], cwd=REPO_LOCAL_DIR, quiet=False, check=False)
    commit_ok = (commit_res.returncode == 0)

    if not commit_ok:
        print("Commit failed.")
        return {
            "committed": False,
            "pushed": False,
            "reason": "commit_failed"
        }

    if not GITHUB_TOKEN:
        print("Commit succeeded locally. Push skipped because no GitHub token was provided.")
        return {
            "committed": True,
            "pushed": False,
            "reason": "no_token"
        }

    push_res = git_authenticated_cmd(["push", "origin", gh["branch"]], token=GITHUB_TOKEN, cwd=REPO_LOCAL_DIR, quiet=False, check=False)
    push_ok = (push_res.returncode == 0)

    if push_ok:
        print("Push completed successfully.")
    else:
        print("Commit succeeded, but push failed.")

    return {
        "committed": True,
        "pushed": push_ok,
        "reason": "ok" if push_ok else "push_failed"
    }

# -----------------------------
# 0.12 Analysis template lock
# -----------------------------
ANALYSIS_TEMPLATE = {
    "primary_reference_template": "Uploaded PDF statistical style",
    "ignore_for_statistical_decisions": "Methodology.docx ANOVA/LSD workflow",
    "time_scale_in_actual_workbook": ["24 HAS", "48 HAS", "72 HAS"],
    "baseline_column_prefix": "BS_",
    "life_stages": ["Egg", "Nymph", "Adult"],
    "mortality_correction": "Abbott corrected mortality",
    "assumption_tests": ["Shapiro-Wilk", "Levene"],
    "group_comparison": "Kruskal-Wallis + Conover-Iman post-hoc + Holm-Bonferroni adjustment",
    "factorial_nonparametric_model": "ART ANOVA",
    "dose_response_timepoint": "72 HAS",
    "dose_response_model": "4-parameter log-logistic (LL.4), fallback LL.3 if lower asymptote unstable",
    "multivariate_analysis": "PCA + KMO at 72 HAS",
    "table_style": "Mean ± SE with grouping letters",
    "figure_export": "PNG only",
    "table_export": "XLSX only",
    "dpi": 300,
    "output_excel": "Final_Analysis_Mishu.xlsx"
}

print("\nLocked analysis template:")
for k, v in ANALYSIS_TEMPLATE.items():
    print(f" - {k}: {v}")

# -----------------------------
# 0.13 Read workbook
# -----------------------------
xlsx = pd.ExcelFile(TARGET_XLSX)
sheet_names = xlsx.sheet_names

print("\nSheets found in workbook:")
for s in sheet_names:
    print(" -", s)

required_columns = [
    "Conc", "Rep",
    "BS_Egg", "24_Egg", "48_Egg", "72_Egg",
    "BS_Nymph", "24_Nymph", "48_Nymph", "72_Nymph",
    "BS_Adult", "24_Adult", "48_Adult", "72_Adult"
]

def normalize_sheet_name(sheet_name: str) -> str:
    s = str(sheet_name).strip()
    s = re.sub(r"\s+", " ", s)
    if s == "Tundra 20SP":
        s = "Tundra 20 SP"
    if s == "Ravzoom 14.5 SC":
        s = "Rav-zoom 14.5 SC"
    return s

def extract_concentration_numeric(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s == "control":
        return 0.0
    s = s.replace("ppm", "").strip()
    m = re.search(r"[-+]?\d*\.?\d+", s)
    return float(m.group()) if m else np.nan

def classify_treatment_label(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    return "control" if s == "control" else "treated"

# -----------------------------
# 0.14 Workbook audit + cleaned copy
# -----------------------------
audit_rows = []
cleaned_preview = {}
cleaned_sheets = {}
any_repairs = False

for raw_sheet_name in sheet_names:
    df = pd.read_excel(TARGET_XLSX, sheet_name=raw_sheet_name)

    missing_required = [c for c in required_columns if c not in df.columns]
    conc_missing_before = int(df["Conc"].isna().sum()) if "Conc" in df.columns else np.nan

    if "Conc" in df.columns:
        if df["Conc"].isna().sum() > 0:
            df["Conc"] = df["Conc"].ffill()
            any_repairs = True
        df["Conc"] = df["Conc"].astype(str).str.strip()
        df["Conc"] = df["Conc"].replace({"nan": np.nan})
        df["Treatment_Type"] = df["Conc"].apply(classify_treatment_label)
        df["Conc_ppm"] = df["Conc"].apply(extract_concentration_numeric)

    conc_missing_after = int(df["Conc"].isna().sum()) if "Conc" in df.columns else np.nan
    normalized_name = normalize_sheet_name(raw_sheet_name)

    if "Conc" in df.columns:
        control_mask = df["Treatment_Type"].eq("control")
        control_reps = int(control_mask.sum())
        treated_levels = int(df.loc[~control_mask, "Conc_ppm"].dropna().nunique())
        treated_conc_list = sorted(df.loc[~control_mask, "Conc_ppm"].dropna().unique().tolist())
    else:
        control_reps = np.nan
        treated_levels = np.nan
        treated_conc_list = []

    numeric_cols = [c for c in df.columns if c not in ["Conc", "Rep", "Treatment_Type"]]
    numeric_missing = int(df[numeric_cols].isna().sum().sum()) if numeric_cols else 0

    audit_rows.append({
        "Raw_Sheet_Name": raw_sheet_name,
        "Standardized_Sheet_Name": normalized_name,
        "Rows": int(df.shape[0]),
        "Columns": int(df.shape[1]),
        "Missing_Required_Columns": ", ".join(missing_required) if missing_required else "",
        "Conc_Missing_Before_FFill": conc_missing_before,
        "Conc_Missing_After_FFill": conc_missing_after,
        "Control_Replicates": control_reps,
        "Treated_Levels": treated_levels,
        "Treated_Concentrations_ppm": ", ".join([str(x) for x in treated_conc_list]),
        "Other_Numeric_Missing_Cells": numeric_missing,
        "Structure_OK": len(missing_required) == 0
    })

    cleaned_preview[normalized_name] = df.head(8).copy()
    cleaned_sheets[raw_sheet_name] = df.copy()

audit_df = pd.DataFrame(audit_rows)

display(Markdown("## Step 0 workbook audit"))
display(audit_df)

for sname, preview_df in cleaned_preview.items():
    display(Markdown(f"### Preview: {sname}"))
    display(preview_df)

if not audit_df["Structure_OK"].all():
    bad_sheets = audit_df.loc[~audit_df["Structure_OK"], "Raw_Sheet_Name"].tolist()
    raise ValueError("Workbook structure mismatch in these sheet(s): " + ", ".join(bad_sheets))

# Optional cleaned workbook copy
cleaned_copy_saved = False
if SAVE_CLEANED_WORKBOOK_COPY and any_repairs:
    with pd.ExcelWriter(CLEANED_WORKBOOK_COPY, engine="openpyxl") as writer:
        for raw_sheet_name in sheet_names:
            cleaned_sheets[raw_sheet_name].drop(columns=["Treatment_Type", "Conc_ppm"], errors="ignore").to_excel(
                writer, sheet_name=raw_sheet_name, index=False
            )
    cleaned_copy_saved = True
    print(f"\nCleaned workbook copy saved:\n{CLEANED_WORKBOOK_COPY}")

# -----------------------------
# 0.15 Save Step 0 outputs
# -----------------------------
with pd.ExcelWriter(TARGET_AUDIT_XLSX, engine="openpyxl") as writer:
    audit_df.to_excel(writer, sheet_name="audit_summary", index=False)
    for sname, preview_df in cleaned_preview.items():
        preview_df.to_excel(writer, sheet_name=sname[:31], index=False)

project_metadata = {
    "project_root": str(PROJECT_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "target_workbook": str(TARGET_XLSX),
    "github_notebook_url": github_url,
    "github_url_source": github_source_file,
    "github_repo_owner": gh["owner"],
    "github_repo_name": gh["repo"],
    "github_branch": gh["branch"],
    "github_repo_url": gh["repo_url"],
    "github_raw_notebook_url": gh["raw_url"],
    "repo_local_dir": str(REPO_LOCAL_DIR),
    "repo_outputs_dir": str(REPO_OUTPUTS_DIR),
    "repo_notebook_path": str(NOTEBOOK_REPO_PATH),
    "push_enabled": bool(GITHUB_TOKEN),
    "git_user_name": GIT_USER_NAME,
    "git_user_email": GIT_USER_EMAIL,
    "sheet_names_raw": sheet_names,
    "sheet_names_standardized": [normalize_sheet_name(s) for s in sheet_names],
    "analysis_template": ANALYSIS_TEMPLATE,
    "required_columns": required_columns,
    "cleaned_workbook_copy_saved": cleaned_copy_saved,
    "cleaned_workbook_copy_path": str(CLEANED_WORKBOOK_COPY) if cleaned_copy_saved else None
}

with open(TARGET_META_JSON, "w", encoding="utf-8") as f:
    json.dump(project_metadata, f, indent=2, ensure_ascii=False)

repo_audit_xlsx = STEP0_REPO_DIR / "Step0_Workbook_Audit.xlsx"
repo_meta_json = STEP0_REPO_DIR / "Step0_Project_Metadata.json"

audit_df.to_excel(repo_audit_xlsx, index=False)

with open(repo_meta_json, "w", encoding="utf-8") as f:
    json.dump(project_metadata, f, indent=2, ensure_ascii=False)

if cleaned_copy_saved:
    shutil.copy2(CLEANED_WORKBOOK_COPY, STEP0_REPO_DIR / "Step0_Cleaned_Workbook_Copy.xlsx")

save_text_to_repo(
    text=(
        "Step 0 completed.\n"
        f"Workbook: {TARGET_XLSX}\n"
        f"GitHub notebook URL: {github_url}\n"
        f"Push enabled: {bool(GITHUB_TOKEN)}\n"
        f"Git user name: {GIT_USER_NAME}\n"
        f"Git user email: {GIT_USER_EMAIL}\n"
        f"Cleaned workbook copy saved: {cleaned_copy_saved}\n"
        "Tables: XLSX only\n"
        "Figures: PNG only\n"
    ),
    step_number=0,
    filename="Step0_Summary.txt"
)

# -----------------------------
# 0.16 Verify metadata JSON
# -----------------------------
with open(TARGET_META_JSON, "r", encoding="utf-8") as f:
    meta_check = json.load(f)

print("\nMetadata JSON verification:")
print("github_notebook_url =", meta_check.get("github_notebook_url"))
print("github_url_source   =", meta_check.get("github_url_source"))
print("repo_local_dir      =", meta_check.get("repo_local_dir"))
print("repo_outputs_dir    =", meta_check.get("repo_outputs_dir"))
print("repo_notebook_path  =", meta_check.get("repo_notebook_path"))
print("push_enabled        =", meta_check.get("push_enabled"))
print("git_user_name       =", meta_check.get("git_user_name"))
print("git_user_email      =", meta_check.get("git_user_email"))
print("cleaned_copy_saved  =", meta_check.get("cleaned_workbook_copy_saved"))

# -----------------------------
# 0.17 Optional automatic Step 0 commit/push
# -----------------------------
if AUTO_PUSH_STEP0:
    add_list = [STEP0_REPO_DIR, NOTEBOOK_REPO_PATH]
    step0_git_result = git_commit_push(
        message="Step 0: initialize workbook audit and GitHub sync workflow",
        add_paths=add_list
    )
else:
    step0_git_result = {
        "committed": False,
        "pushed": False,
        "reason": "auto_push_disabled"
    }

print("\nStep 0 git result:")
print(step0_git_result)

# -----------------------------
# 0.18 Final summary
# -----------------------------
print("\n" + "="*72)
print("STEP 0 COMPLETED SUCCESSFULLY")
print("="*72)
print(f"Workbook used           : {TARGET_XLSX}")
print(f"Audit file saved        : {TARGET_AUDIT_XLSX}")
print(f"Metadata file saved     : {TARGET_META_JSON}")
print(f"GitHub notebook URL     : {github_url}")
print(f"GitHub URL source       : {github_source_file}")
print(f"Local repo path         : {REPO_LOCAL_DIR}")
print(f"Repo notebook path      : {NOTEBOOK_REPO_PATH}")
print(f"Repo outputs root       : {REPO_OUTPUTS_DIR}")
print(f"Push enabled            : {bool(GITHUB_TOKEN)}")
print(f"Git user name           : {GIT_USER_NAME}")
print(f"Git user email          : {GIT_USER_EMAIL}")
print(f"Cleaned workbook copy   : {CLEANED_WORKBOOK_COPY if cleaned_copy_saved else 'Not needed'}")

print("\nKey findings:")
print("- Workbook contains 6 insecticide sheets.")
print("- Real time points are 24, 48, and 72 HAS.")
print("- Ravzoom missingness is checked dynamically, not assumed.")
print("- Metadata JSON stores GitHub and git-user details.")
print("- Token is not stored in the git remote URL.")
print("- Step outputs can be pushed to GitHub after every step if repo write permission is valid.")
print("- Tables are saved as XLSX only.")
print("- Figures are saved as PNG only.")
print("- The dataset is ready for Step 1 preprocessing and Abbott correction.")

print("\nAvailable helper functions after Step 0:")
print(" - save_df_to_repo(df, step_number, filename, index=False)")
print(" - save_text_to_repo(text, step_number, filename)")
print(" - save_figure_to_repo(fig, step_number, stem, dpi=300)")
print(" - git_status()")
print(" - git_commit_push(message, add_paths=None)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root : /content/drive/MyDrive/Data Analysis March 2026
Output folder: /content/drive/MyDrive/Data Analysis March 2026/Mishu

Candidate Excel files found:
 - /content/drive/MyDrive/Data Analysis March 2026/Aleurodicus_rugioperculatus.xlsx
 - /content/drive/MyDrive/Data Analysis March 2026/Mishu/Aleurodicus_rugioperculatus.xlsx
 - /content/drive/MyDrive/CORRELATION AND P VALUE.xlsx
 - /content/drive/MyDrive/New Microsoft Excel Worksheet.xlsx
 - /content/drive/MyDrive/temporary upload/to_run.xlsx
 - /content/drive/MyDrive/temporary upload/Akib Main Data.xlsx
 - /content/drive/MyDrive/temporary upload/NILA Main Data.xlsx
 - /content/drive/MyDrive/temporary upload/Analyzed Data/Himel/himel.xlsx
 - /content/drive/MyDrive/temporary upload/Analyzed Data/Sadia/sadia.xlsx
 - /content/drive/MyDrive/temporary upload/Analyzed Data/Shormi/shormi.xlsx
 - /content/dr

## Step 0 workbook audit

,Raw_Sheet_Name,Standardized_Sheet_Name,Rows,Columns,Missing_Required_Columns,Conc_Missing_Before_FFill,Conc_Missing_After_FFill,Control_Replicates,Treated_Levels,Treated_Concentrations_ppm,Other_Numeric_Missing_Cells,Structure_OK
0,Tundra 20SP,Tundra 20 SP,28,16,,0,0,4,6,"2.0, 4.0, 8.0, 16.0, 32.0, 64.0",0,True
1,Actara 25 WG,Actara 25 WG,28,16,,0,0,4,6,"1.25, 2.5, 5.0, 10.0, 20.0, 40.0",0,True
2,Tiddo 20 SL,Tiddo 20 SL,28,16,,0,0,4,6,"1.0, 2.0, 4.0, 8.0, 16.0, 32.0",0,True
3,Shobicron 425 EC,Shobicron 425 EC,28,16,,0,0,4,6,"8.5, 17.0, 34.0, 68.0, 136.0, 272.0",0,True
4,Panel 50 SP,Panel 50 SP,28,16,,0,0,4,6,"5.0, 10.0, 20.0, 40.0, 80.0, 160.0",0,True
5,Ravzoom 14.5 SC,Rav-zoom 14.5 SC,28,16,,0,0,4,6,"1.015, 2.03, 4.06, 8.12, 16.24, 32.48",0,True


### Preview: Tundra 20 SP

,Conc,Rep,BS_Egg,24_Egg,48_Egg,72_Egg,BS_Nymph,24_Nymph,48_Nymph,72_Nymph,BS_Adult,24_Adult,48_Adult,72_Adult,Treatment_Type,Conc_ppm
0,control,R1,58,55,50,44,54,51,45,39,12,10,7,5,control,0.0
1,control,R2,49,45,40,34,86,82,77,71,14,12,9,5,control,0.0
2,control,R3,76,72,67,61,75,71,65,58,8,7,5,3,control,0.0
3,control,R4,91,88,83,77,65,61,56,50,9,6,3,1,control,0.0
4,2ppm,R1,89,53,28,4,72,40,20,4,13,8,3,0,treated,2.0
5,2ppm,R2,72,60,34,18,71,59,43,26,15,12,7,0,treated,2.0
6,2ppm,R3,76,64,48,31,59,47,32,15,8,5,1,0,treated,2.0
7,2ppm,R4,58,47,33,17,57,45,30,13,9,6,2,0,treated,2.0


...

### Preview: Panel 50 SP

,Conc,Rep,BS_Egg,24_Egg,48_Egg,72_Egg,BS_Nymph,24_Nymph,48_Nymph,72_Nymph,BS_Adult,24_Adult,48_Adult,72_Adult,Treatment_Type,Conc_ppm
0,control,R1,57,54,50,44,53,50,46,41,12,10,9,7,control,0.0
1,control,R2,28,25,20,14,43,41,36,30,10,9,9,8,control,0.0
2,control,R3,75,72,68,61,91,88,84,78,7,5,4,4,control,0.0
3,control,R4,38,35,31,25,30,27,23,16,9,7,5,3,control,0.0
4,5ppm,R1,68,48,41,28,78,49,44,39,15,13,10,6,treated,5.0
5,5ppm,R2,76,73,69,62,69,64,58,51,16,10,5,2,treated,5.0
6,5ppm,R3,75,71,65,59,56,52,47,40,14,9,5,1,treated,5.0
7,5ppm,R4,60,56,51,45,91,85,78,70,12,7,3,0,treated,5.0


### Preview: Rav-zoom 14.5 SC

,Conc,Rep,BS_Egg,24_Egg,48_Egg,72_Egg,BS_Nymph,24_Nymph,48_Nymph,72_Nymph,BS_Adult,24_Adult,48_Adult,72_Adult,Treatment_Type,Conc_ppm
0,control,R1,59,56,52,47,54,50,45,39,13,11,8,5,control,0.000
1,control,R2,30,27,23,19,48,45,39,33,8,7,5,3,control,0.000
2,control,R3,76,72,68,63,92,88,83,76,11,9,6,3,control,0.000
3,control,R4,42,38,33,28,33,29,24,19,9,6,3,2,control,0.000
4,1.015ppm,R1,51,41,36,29,61,35,33,30,14,10,9,9,treated,1.015
5,1.015ppm,R2,48,46,42,37,57,53,48,41,15,8,5,2,treated,1.015
6,1.015ppm,R3,65,62,58,52,79,73,66,58,16,7,3,1,treated,1.015
7,1.015ppm,R4,44,42,39,34,75,69,62,55,13,6,3,1,treated,1.015


Saved text to: /content/Aleurodicus-rugioperculatus/analysis_outputs/step_00/Step0_Summary.txt

Metadata JSON verification:
github_notebook_url = https://github.com/hemel5612-lgtm/Aleurodicus-rugioperculatus/blob/main/Aleurodicus_rugioperculatus.ipynb
github_url_source   = interactive_input
repo_local_dir      = /content/Aleurodicus-rugioperculatus
repo_outputs_dir    = /content/Aleurodicus-rugioperculatus/analysis_outputs
repo_notebook_path  = /content/Aleurodicus-rugioperculatus/Aleurodicus_rugioperculatus.ipynb
push_enabled        = True
git_user_name       = hemel5612-lgtm
git_user_email      = hemel5612@gmail.com
cleaned_copy_saved  = False
[main 5e35616] Step 0: initialize workbook audit and GitHub sync workflow
 1 file changed, 0 insertions(+), 0 deletions(-)
To https://github.com/hemel5612-lgtm/Aleurodicus-rugioperculatus.git
 ! [rejected]        main -> main (non-fast-forward)
error: failed to push some refs to 'https://github.com/hemel5612-lgtm/Aleurodicus-rugioperculatus.git

In [2]:
# ============================================================
# STEP 1 — Merge all sheets and compute raw + Abbott mortality
# Exports:
#   - XLSX tables to analysis_outputs/step_01/
#   - notes txt to analysis_outputs/step_01/
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# -----------------------------
# 1.1 Parameters
# -----------------------------
TIMEPOINTS = ["24", "48", "72"]
STAGES = ["Egg", "Nymph", "Adult"]

# -----------------------------
# 1.2 Read and standardize all sheets
# -----------------------------
master_parts = []

for raw_sheet_name in xlsx.sheet_names:
    df = pd.read_excel(TARGET_XLSX, sheet_name=raw_sheet_name).copy()

    # Standardize treatment information
    df["Conc"] = df["Conc"].astype(str).str.strip()
    df["Treatment_Type"] = df["Conc"].str.lower().eq("control").map(
        {True: "control", False: "treated"}
    )
    df["Conc_ppm"] = df["Conc"].apply(extract_concentration_numeric)
    df["Insecticide"] = normalize_sheet_name(raw_sheet_name)

    master_parts.append(df)

master_df = pd.concat(master_parts, ignore_index=True)

display(Markdown("## Step 1A. Master merged data"))
display(master_df.head(12))

print("Master dataframe shape:", master_df.shape)
print("Insecticides:", master_df["Insecticide"].nunique())
print("Treatment types:", master_df["Treatment_Type"].value_counts().to_dict())

# -----------------------------
# 1.3 Convert wide data to long format
# -----------------------------
long_rows = []

for _, row in master_df.iterrows():
    for stage in STAGES:
        baseline = row[f"BS_{stage}"]

        for tp in TIMEPOINTS:
            observed = row[f"{tp}_{stage}"]

            if pd.isna(baseline) or baseline == 0:
                raw_mortality = np.nan
            else:
                raw_mortality = ((baseline - observed) / baseline) * 100

            long_rows.append({
                "Insecticide": row["Insecticide"],
                "Conc": row["Conc"],
                "Conc_ppm": row["Conc_ppm"],
                "Rep": row["Rep"],
                "Treatment_Type": row["Treatment_Type"],
                "Stage": stage,
                "Time_HAS": int(tp),
                "Baseline_Count": baseline,
                "Observed_Count": observed,
                "Raw_Mortality_pct": raw_mortality
            })

long_df = pd.DataFrame(long_rows)

display(Markdown("## Step 1B. Long-format mortality data"))
display(long_df.head(18))

print("Long dataframe shape:", long_df.shape)

# -----------------------------
# 1.4 Compute control mortality means
# Abbott correction:
# Corrected = ((T - C) / (100 - C)) * 100
# where T = treatment raw mortality, C = control raw mortality
# -----------------------------
control_means = (
    long_df[long_df["Treatment_Type"] == "control"]
    .groupby(["Insecticide", "Stage", "Time_HAS"], as_index=False)["Raw_Mortality_pct"]
    .mean()
    .rename(columns={"Raw_Mortality_pct": "Control_Mortality_Mean_pct"})
)

long_df = long_df.merge(
    control_means,
    on=["Insecticide", "Stage", "Time_HAS"],
    how="left"
)

def abbott_corrected(t, c):
    if pd.isna(t) or pd.isna(c):
        return np.nan
    if (100 - c) == 0:
        return np.nan
    value = ((t - c) / (100 - c)) * 100
    return max(0, min(100, value))

long_df["Abbott_Corrected_Mortality_pct"] = np.where(
    long_df["Treatment_Type"].eq("treated"),
    long_df.apply(
        lambda r: abbott_corrected(
            r["Raw_Mortality_pct"],
            r["Control_Mortality_Mean_pct"]
        ),
        axis=1
    ),
    np.nan
)

# -----------------------------
# 1.5 Create summary table for checking
# FIXED:
# n now counts replicates, so controls also show n = 4
# -----------------------------
step1_summary = (
    long_df.groupby(
        ["Insecticide", "Treatment_Type", "Conc", "Conc_ppm", "Stage", "Time_HAS"],
        as_index=False
    )
    .agg(
        n=("Rep", "count"),
        Baseline_Mean=("Baseline_Count", "mean"),
        Observed_Mean=("Observed_Count", "mean"),
        Raw_Mortality_Mean_pct=("Raw_Mortality_pct", "mean"),
        Raw_Mortality_SE_pct=(
            "Raw_Mortality_pct",
            lambda x: x.std(ddof=1) / np.sqrt(x.count()) if x.count() > 1 else np.nan
        ),
        Control_Mortality_Mean_pct=("Control_Mortality_Mean_pct", "mean"),
        Abbott_Mean_pct=("Abbott_Corrected_Mortality_pct", "mean"),
        Abbott_SE_pct=(
            "Abbott_Corrected_Mortality_pct",
            lambda x: x.std(ddof=1) / np.sqrt(x.count()) if x.count() > 1 else np.nan
        ),
    )
    .sort_values(["Insecticide", "Stage", "Time_HAS", "Treatment_Type", "Conc_ppm"])
    .reset_index(drop=True)
)

display(Markdown("## Step 1C. Summary check table"))
display(step1_summary.head(24))

# -----------------------------
# 1.6 Quick validation checks
# -----------------------------
print("\nValidation checks:")
print("- Unique insecticides:", master_df['Insecticide'].nunique())
print("- Expected master rows = 6 sheets x 28 rows = 168")
print("- Actual master rows  =", len(master_df))
print("- Expected long rows  = 168 x 3 stages x 3 times = 1512")
print("- Actual long rows    =", len(long_df))
print("- Control n values    =", sorted(step1_summary.loc[step1_summary['Treatment_Type'] == 'control', 'n'].unique().tolist()))
print("- Treated n values    =", sorted(step1_summary.loc[step1_summary['Treatment_Type'] == 'treated', 'n'].unique().tolist()))

# -----------------------------
# 1.7 Export Step 1 outputs to repo
# -----------------------------
save_df_to_repo(master_df, step_number=1, filename="Table_01_Master_Merged_Data.xlsx", index=False)
save_df_to_repo(long_df, step_number=1, filename="Table_02_Long_Mortality_Data.xlsx", index=False)
save_df_to_repo(control_means, step_number=1, filename="Table_03_Control_Mortality_Means.xlsx", index=False)
save_df_to_repo(step1_summary, step_number=1, filename="Table_04_Step1_Summary_Check.xlsx", index=False)

save_text_to_repo(
    text=(
        "Step 1 completed.\n"
        "Merged all six insecticide sheets.\n"
        "Converted wide data to long format.\n"
        "Computed raw mortality and Abbott-corrected mortality.\n"
        "Corrected summary table replicate count so both control and treated groups show n = 4.\n"
        "Ready for Step 2 assumption checks and nonparametric group comparisons.\n"
    ),
    step_number=1,
    filename="Step1_Notes.txt"
)

# -----------------------------
# 1.8 Push Step 1 to GitHub
# -----------------------------
step1_git_result = git_commit_push(
    "Step 1: merge sheets and compute Abbott-corrected mortality",
    add_paths=[
        REPO_OUTPUTS_DIR / "step_01",
        NOTEBOOK_REPO_PATH
    ]
)

print("\nStep 1 git result:")
print(step1_git_result)

## Step 1A. Master merged data

,Conc,Rep,BS_Egg,24_Egg,48_Egg,72_Egg,BS_Nymph,24_Nymph,48_Nymph,72_Nymph,BS_Adult,24_Adult,48_Adult,72_Adult,Treatment_Type,Conc_ppm,Insecticide
0,control,R1,58,55,50,44,54,51,45,39,12,10,7,5,control,0.0,Tundra 20 SP
1,control,R2,49,45,40,34,86,82,77,71,14,12,9,5,control,0.0,Tundra 20 SP
2,control,R3,76,72,67,61,75,71,65,58,8,7,5,3,control,0.0,Tundra 20 SP
3,control,R4,91,88,83,77,65,61,56,50,9,6,3,1,control,0.0,Tundra 20 SP
4,2ppm,R1,89,53,28,4,72,40,20,4,13,8,3,0,treated,2.0,Tundra 20 SP
5,2ppm,R2,72,60,34,18,71,59,43,26,15,12,7,0,treated,2.0,Tundra 20 SP
6,2ppm,R3,76,64,48,31,59,47,32,15,8,5,1,0,treated,2.0,Tundra 20 SP
7,2ppm,R4,58,47,33,17,57,45,30,13,9,6,2,0,treated,2.0,Tundra 20 SP
8,4ppm,R1,67,33,18,3,68,37,20,3,12,5,3,1,treated,4.0,Tundra 20 SP
9,4ppm,R2,63,46,25,7,57,42,24,4,11,6,1,0,treated,4.0,Tundra 20 SP


Master dataframe shape: (168, 17)
Insecticides: 6
Treatment types: {'treated': 144, 'control': 24}


## Step 1B. Long-format mortality data

,Insecticide,Conc,Conc_ppm,Rep,Treatment_Type,Stage,Time_HAS,Baseline_Count,Observed_Count,Raw_Mortality_pct
0,Tundra 20 SP,control,0.0,R1,control,Egg,24,58,55,5.172414
1,Tundra 20 SP,control,0.0,R1,control,Egg,48,58,50,13.793103
2,Tundra 20 SP,control,0.0,R1,control,Egg,72,58,44,24.137931
3,Tundra 20 SP,control,0.0,R1,control,Nymph,24,54,51,5.555556
4,Tundra 20 SP,control,0.0,R1,control,Nymph,48,54,45,16.666667
5,Tundra 20 SP,control,0.0,R1,control,Nymph,72,54,39,27.777778
6,Tundra 20 SP,control,0.0,R1,control,Adult,24,12,10,16.666667
7,Tundra 20 SP,control,0.0,R1,control,Adult,48,12,7,41.666667
8,Tundra 20 SP,control,0.0,R1,control,Adult,72,12,5,58.333333
9,Tundra 20 SP,control,0.0,R2,control,Egg,24,49,45,8.163265


Long dataframe shape: (1512, 10)


## Step 1C. Summary check table

,Insecticide,Treatment_Type,Conc,Conc_ppm,Stage,Time_HAS,n,Baseline_Mean,Observed_Mean,Raw_Mortality_Mean_pct,Raw_Mortality_SE_pct,Control_Mortality_Mean_pct,Abbott_Mean_pct,Abbott_SE_pct
0,Actara 25 WG,control,control,0.00,Adult,24,4,11.25,9.00,20.197164,2.128973,20.197164,NaN,NaN
1,Actara 25 WG,treated,1.25ppm,1.25,Adult,24,4,10.00,7.00,30.776515,2.848790,20.197164,13.256861,3.569785
2,Actara 25 WG,treated,2.5ppm,2.50,Adult,24,4,12.00,6.50,48.251748,7.621938,20.197164,35.154871,9.550961
3,Actara 25 WG,treated,5ppm,5.00,Adult,24,4,11.00,4.25,67.817460,10.435904,20.197164,59.672436,13.077110
4,Actara 25 WG,treated,10ppm,10.00,Adult,24,4,14.50,4.25,73.647661,9.351465,20.197164,66.978192,11.718211
5,Actara 25 WG,treated,20ppm,20.00,Adult,24,4,13.00,3.50,74.468730,13.077048,20.197164,68.007064,16.386696
6,Actara 25 WG,treated,40ppm,40.00,Adult,24,4,12.50,2.00,83.155806,9.749911,20.197164,78.892738,12.217499
7,Actara 25 WG,control,control,0.00,Adult,48,4,11.25,5.50,51.218920,5.869782,51.218920,NaN,NaN
8,Actara 25 WG,treated,1.25ppm,1.25,Adult,48,4,10.00,3.50,66.256313,7.276894,51.218920,30.826282,14.917451
9,Actara 25 WG,treated,2.5ppm,2.50,Adult,48,4,12.00,3.00,75.695416,5.216760,51.218920,50.176207,10.694229



Validation checks:
- Unique insecticides: 6
- Expected master rows = 6 sheets x 28 rows = 168
- Actual master rows  = 168
- Expected long rows  = 168 x 3 stages x 3 times = 1512
- Actual long rows    = 1512
- Control n values    = [4]
- Treated n values    = [4]
Saved table to: /content/Aleurodicus-rugioperculatus/analysis_outputs/step_01/Table_01_Master_Merged_Data.xlsx
Saved table to: /content/Aleurodicus-rugioperculatus/analysis_outputs/step_01/Table_02_Long_Mortality_Data.xlsx
Saved table to: /content/Aleurodicus-rugioperculatus/analysis_outputs/step_01/Table_03_Control_Mortality_Means.xlsx
Saved table to: /content/Aleurodicus-rugioperculatus/analysis_outputs/step_01/Table_04_Step1_Summary_Check.xlsx
Saved text to: /content/Aleurodicus-rugioperculatus/analysis_outputs/step_01/Step1_Notes.txt
[main ff22af7] Step 1: merge sheets and compute Abbott-corrected mortality
 4 files changed, 0 insertions(+), 0 deletions(-)
To https://github.com/hemel5612-lgtm/Aleurodicus-rugioperculatus.gi